In [1]:
import os 
import certifi 
from dotenv import load_dotenv

#from langchain_openai import ChatOpenAI
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub
from langchain.tools import tool
import requests

In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
# ==========================================
# LOAD ENV VARIABLES
# ==========================================
os.environ["SSL_CERT_FILE"] = certifi.where()
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

In [4]:
search_tool = TavilySearchResults(max_results=2)

In [5]:
@tool
def get_weather_data(city: str) -> str:
    """
    Fetch current weather information for a city.
    """

    url = (
        f"https://api.weatherstack.com/current?"
        f"access_key={WEATHERSTACK_API_KEY}&query={city}"
    )

    response = requests.get(url)

    data = response.json()

    if "current" not in data:
        return f"Could not fetch weather data for {city}"

    return (
        f"City: {city}\n"
        f"Temperature: {data['current']['temperature']}°C\n"
        f"Weather: {data['current']['weather_descriptions'][0]}\n"
        f"Humidity: {data['current']['humidity']}%"
    )

In [6]:
result = search_tool.invoke("Give me the latest news on AI")
result

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'AI in Action\n\nJuly 21, 2026\n\n### AWS GraphRAG deployment cuts drug research cycles by 87%\n\nHealthcare & Wellness AI\n\nJuly 9, 2026\n\n### Anthropic deploys Claude Sonnet 5, Fable and Mythos restored\n\nInside AI\n\nJuly 1, 2026\n\n#### Industries\n\n### Google’s Gemini 3.6 Flash targets enterprise agent token costs\n\nAI in Action\n\nJuly 21, 2026\n\n### HP accelerates enterprise workflows with OpenAI Frontier\n\nWorld of Work\n\nJune 29, 2026\n\n### Omio scales travel product development using OpenAI models\n\nAI in Action\n\nJune 23, 2026\n\n#### Deep Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026\n\n### China’s AI just mapped its entire renewable energy grid. Here’s why the rest of the world should pay attention\n\nEnvironment & Sustainability\n\nMay 22, 2026 [...] AI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majora

In [7]:
# ==========================================
# LLM
# ==========================================

# llm = ChatOpenAI(
#     model="gpt-3.5-turbo",
#     temperature=0,
#     api_key=OPENAI_API_KEY
# )

llm = ChatGroq(
    model="llama-3.3-70b-versatile", 
    temperature=0,
    api_key=GROQ_API_KEY
)

In [8]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='Why did the AI program go on a diet?\n\nBecause it wanted to lose some bytes! (get it?)', response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 41, 'total_tokens': 64, 'completion_time': 0.062820631, 'completion_tokens_details': None, 'prompt_time': 0.003936436, 'prompt_tokens_details': None, 'queue_time': 0.16259644, 'total_time': 0.066757067}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'finish_reason': 'stop', 'logprobs': None}, id='run-8a7eb246-c40e-45a7-8476-df343987aa2d-0')

In [9]:
# ==========================================
# PROMPT
# ==========================================

prompt = hub.pull("hwchase17/react")

c:\anaconda3\envs\GenAI\Lib\site-packages\langchain\hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [10]:
prompt

PromptTemplate(input_variables=['agent_scratchpad', 'input', 'tool_names', 'tools'], metadata={'lc_hub_owner': 'hwchase17', 'lc_hub_repo': 'react', 'lc_hub_commit_hash': 'd15fe3c426f1c4b3f37c9198853e4a86e20c425ca7f4752ec0c9b0e97ca7ea4d'}, template='Answer the following questions as best you can. You have access to the following tools:\n\n{tools}\n\nUse the following format:\n\nQuestion: the input question you must answer\nThought: you should always think about what to do\nAction: the action to take, should be one of [{tool_names}]\nAction Input: the input to the action\nObservation: the result of the action\n... (this Thought/Action/Action Input/Observation can repeat N times)\nThought: I now know the final answer\nFinal Answer: the final answer to the original input question\n\nBegin!\n\nQuestion: {input}\nThought:{agent_scratchpad}')

In [11]:
# ==========================================
# TOOLS
# ==========================================

tools = [search_tool, get_weather_data]

In [12]:
# ==========================================
# CREATE AGENT
# ==========================================

agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

In [13]:
# ==========================================
# EXECUTOR
# ==========================================

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True
)

In [14]:
# ==========================================
# RUN
# ==========================================

response = agent_executor.invoke({
    "input": (
        "Find the capital of India"
        "and then find its current weather."
    )
})



> Entering new AgentExecutor chain...


c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_

Thought: To find the capital of India and its current weather, I first need to identify the capital of India. I can use the tavily_search_results_json tool to search for this information.

Action: tavily_search_results_json
Action Input: "capital of India"

c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()


[{'url': 'https://en.wikipedia.org/wiki/List_of_capitals_of_India', 'content': '## Modern period\n\n[edit]\n\n In 1858, Allahabad the capital of India for a day when it also served as the capital of North-Western Provinces.\n During the British Raj, until 1911, Calcutta was the capital of India.\n By the latter half of the 19th century, Shimla had become the summer capital.\n King George V proclaimed the transfer of the capital from Calcutta to Delhi at the climax of the 1911 Delhi Durbar on 12 December 1911. The buildings housing the Viceroy, government, and parliament were inaugurated in early 1931.\n\n## References\n\n[edit] [...] Appearance\n\nFrom Wikipedia, the free encyclopedia\n\n|  |\n\n| This article needs more citations. Please help improve this article by adding citations to reliable sources. Unsourced material may be challenged and removed. Find sources: "List of capitals of India" – news · newspapers · books · scholar · JSTOR (September 2012) (Learn how and when to remove

c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_

I have found that the capital of India is New Delhi. Now, I need to find the current weather in New Delhi. I can use the get_weather_data tool to fetch the current weather information for New Delhi.

Action: get_weather_data
Action Input: "New Delhi"City: New Delhi
Temperature: 33°C
Weather: Mist
Humidity: 75%

c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_

I now know the final answer
Final Answer: The capital of India is New Delhi, and its current weather is 33°C with mist and 75% humidity.

> Finished chain.


c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_groq\chat_models.py:315: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.13/migration/
  chunk = chunk.dict()
c:\anaconda3\envs\GenAI\Lib\site-packages\langchain_

In [15]:
print(response["output"])

The capital of India is New Delhi, and its current weather is 33°C with mist and 75% humidity.
